# <b><u>EDA - Détection de Fraude Chargeback</b>

### <b>Contexte</b>
Nous faisons, ici, l'analyse d'un jeu de données de transactions bancaires pour :
<br>
<li>Comprendre les données : distributions, valeurs manquantes, types de variables
<br>
<li>Identifier les patterns de fraude : quelles caractéristiques distinguent une transaction frauduleuse d'une légitime
<br>
<li>Visualiser les déséquilibres
<br>
<li>Détecter les anomalies : montants inhabituels, fréquences, pays d'origine...

### Imports

In [91]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [48]:
# import dataset
df = pd.read_csv('../data/kaggle_b2_fraud_train_v3.csv')

# 1. COMPRÉHENSION INITIALE DES DONNÉES

Dans le domaine informatique et notamment de la data science, il est impératif de <b>comprendre</b> la donnée que l'on exploite.

## 1.1 Vue d'ensemble

In [ ]:
df.shape

Pour enlever toutes incohérences, il est primordiale de se séparer des valeurs abérantes ou qui ne sont pas interprétables.

### <ol><li>Valeurs abérantes

In [ ]:
import pandas as pd

# Définition de règles plausibles pour certaines colonnes
rules = {
    'age': lambda x: x >= 0,
    'tenure_months': lambda x: x >= 0,
    'annual_income_eur': lambda x: x >= 0,
    'credit_score': lambda x: (x >= 300) & (x <= 850),
    'num_transactions_30d': lambda x: x >= 0,
    'avg_amount_30d_eur': lambda x: x >= 0,
    'max_amount_30d_eur': lambda x: x >= 0,
    'tx_amount_total_30d_eur': lambda x: x >= 0,
    'max_to_avg_ratio': lambda x: x >= 0,
    'days_since_last_login': lambda x: x >= 0,
    'support_tickets_90d': lambda x: x >= 0,
    'chargebacks_12m': lambda x: x >= 0,
    'failed_payments_6m': lambda x: x >= 0,
    'num_devices_30d': lambda x: x >= 0,
    'chargeback_resolution_time_days': lambda x: x >= 0,
}

# Stocker le nombre d'erreurs
impossible_counts = {}

for col, rule in rules.items():
    if col in df.columns:
        mask_invalid = ~df[col].isna() & ~rule(df[col])
        count_invalid = mask_invalid.sum()
        if count_invalid > 0:
            impossible_counts[col] = count_invalid

# Affichage
if impossible_counts:
    print("Colonnes avec valeurs impossibles et nombre d'erreurs :")
    for col, count in impossible_counts.items():
        print(f"{col}: {count} erreur(s)")
else:
    print("Aucune valeur impossible détectée selon les règles définies.")

Colonnes avec valeurs impossibles et nombre d'erreurs :
age: 84 erreur(s)
tenure_months: 203 erreur(s)
annual_income_eur: 117 erreur(s)
avg_amount_30d_eur: 71 erreur(s)


In [ ]:
# suppression des âges négatifs
df = df[df['age'] >= 0]
# suppression des dates de création de compte négatives
df = df[df["tenure_months" ]>= 0]
# suppression des revenus négatifs
df = df[df["annual_income_eur"] >= 0]
# suppression des revenus négatifs
df = df[df["avg_amount_30d_eur"] >= 0]

In [ ]:
df.iloc[:, 0:50].head(2)

,customer_id,account_id,age,tenure_months,annual_income_eur,credit_score,num_transactions_30d,avg_amount_30d_eur,max_amount_30d_eur,days_since_last_login,...,tx_amount_total_30d_eur,max_to_avg_ratio,internal_signal_1,internal_signal_2,internal_signal_3,internal_signal_4,internal_signal_5,internal_signal_6,internal_signal_7,internal_signal_8
0,CUST_6O9Q8D4I36,ACC_TXXXTNEUVKFY,34,108,38635.01,544.0,20,60.92,80.16,4.9,...,1218.40,1.3158,-0.99355,-1.34156,-0.68676,-1.54627,0.39006,0.10963,0.55097,-0.56104
1,CUST_FGUGTW230C,ACC_70VD7A4FFWCW,48,2,19912.97,703.0,21,112.11,571.12,0.3,...,2354.31,5.0943,-0.44874,0.23573,-0.17429,-0.00054,0.03265,-0.40256,0.36218,0.86583


### Typologie de données

In [ ]:
# is_new_device → convertir en int64 (ou bool si 0/1)
df['is_new_device'] = df['is_new_device'].astype('int64')

# signup_source → garder comme object (string)
df['signup_source'] = df['signup_source'].astype('object')

# postal_code → convertir en string pour garder les codes avec zéro en début
df['postal_code'] = df['postal_code'].astype('str')

# days_since_last_login → convertir en int si tu veux uniquement des jours entiers
df['days_since_last_login'] = df['days_since_last_login'].astype('int64')

# 2. ANALYSE DE LA TARGET (Variable Cible)

Quand on fait une analyse aussi poussé que dans notre cas, il faut nécessairement analyser une " <b>Cible</b> "
<br>
Ce qui a pour but d'étudier :
<ul>
    <li> L'équilibre : combien de fraudes vs transactions légitimes ?
    <li> Justifier les models : si déséquilibre alors on utilise un SMOTE (Synthetic Minority Over-sampling Technique)
</ul>
<br>
Concrètement, ça répond à :
<br>
"Est-ce que mon modèle risque d'apprendre à tout prédire comme non-fraude et quand même avoir 99% d'accuracy ?"

## 2.1 Distribution et déséquilibre

In [97]:
# Comptage absolu
df['target_is_fraud'].value_counts(normalize=True)*100

# ==> Rééchantillonnage ou Pondération des classes

target_is_fraud
0    96.917408
1     3.082592
Name: proportion, dtype: float64

## 2.2 Analyse temporelle de la target (si applicable)

In [ ]:
# S'assurer que signup_date est au format datetime
df['signup_date'] = pd.to_datetime(df['signup_date'], errors='coerce')

# Grouper par mois (ou jour) et calculer le taux de fraude
fraud_rate_time = df.groupby(df['signup_date'].dt.to_period('M'))['target_is_fraud'].mean()

# Optionnel : afficher le taux en %
fraud_rate_time_percent = fraud_rate_time * 100
print(fraud_rate_time_percent.head(10))

La fraude reste relativement stable au fil du temps, sans tendance forte à la hausse ou à la baisse.

Aucune variable temporelle explicite n’est nécessaire pour corriger un biais saisonnier, mais il faut garder signup_date si l’on veut éventuellement inclure des features temporelles pour le modèle (ex. mois d’inscription, saison)

### Corrélation avec la feature (dataleakage)

In [105]:
# Sélection des colonnes numériques (excluant la target)
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
num_cols = [col for col in num_cols if col != 'target_is_fraud']

# Calcul de la corrélation de Pearson avec la target
corr_target = df[num_cols + ['target_is_fraud']].corr()['target_is_fraud'].sort_values(ascending=False)

print("Corrélation avec la target :")
print(corr_target.head(7))

chargeback_resolution_time_days    0.676291<br>
post_event_status_code             0.640706<br>
==> les supprimer lors du preprocess

# 3 Valeurs manquantes : 

## 3.1 Quantification

In [112]:
(df.isna().mean() * 100).sort_values(ascending=False).to_frame(name='percent_missing').head(13)

### Missing par ligne : combien de lignes ont 0, 1, 5, 10+ valeurs manquantes ?

In [109]:
missing_per_row = df.isna().sum(axis=1)
missing_per_row.value_counts().sort_index()

In [119]:
colonnes = [
    'partner_risk_indicator', 'legacy_partner_score', 'secondary_email', 
    'region', 'credit_score', 'max_amount_30d_eur', 'device_trust_z', 
    'customer_note', 'ip_risk_z', 'occupation', 'last_ticket_subject', 
    'merchant_category'
]

# Calcul du taux de fraude et de la différence
résumé = []
for col in colonnes:
    taux = df.groupby(df[col].isna())['target_is_fraud'].mean()
    diff = taux.get(True, 0) - taux.get(False, 0)  # missing - non-missing
    résumé.append({
        'diff_missing_present': diff *100
    })

résumé_df = pd.DataFrame(résumé).sort_values('diff_missing_present', key=abs, ascending=False)
print(résumé_df)

Les colonnes last_ticket_subject (-0,78) et max_amount_30d_eur (0,71) portent un signal fort de fraude, mais il est en grande partie corrélé au fait qu’un client passe peu ou aucune commande. Il est donc plus efficace de créer directement une feature nb_commands qui capture cette information, et de supprimer les deux colonnes, qui contiennent en plus beaucoup de valeurs manquantes.

# 4. DOUBLONS

## 4.1 doublons stricts

In [121]:
df.duplicated().sum()

np.int64(6)

In [127]:
dup_customer = df[df.duplicated('customer_id', keep=False)].sort_values('customer_id')
dup_account  = df[df.duplicated('account_id', keep=False)].sort_values('account_id')

print(f"Duplicatas customer_id : {dup_customer.shape[0]}")
print(f"Duplicatas account_id  : {dup_account.shape[0]}")

Duplicatas customer_id : 3780
Duplicatas account_id  : 3780


In [ ]:
mask_dup_customer = df['customer_id'].duplicated(keep=False)

# Appliquer le filtre pour ne garder que ces lignes
df_dup_customer = df[mask_dup_customer]

In [ ]:
df_dup_customer.sort_values(by="customer_id", ascending=False)

In [146]:
# Conserver une ligne aléatoire par customer_id
df_clean = df.groupby('customer_id').sample(n=1, random_state=42).reset_index(drop=True)

print(f"Lignes avant : {df.shape[0]}, lignes après fusion : {df_clean.shape[0]}")

Lignes avant : 139493, lignes après fusion : 137603


# 5. Variables numériques : 

In [ ]:
# Sélectionner les variables numériques
num_cols = df.select_dtypes(include='number').columns

# Matrice de corrélation
corr_matrix = df[num_cols].corr()


# Identifier les corrélations fortes (>0.9)
strong_corr = [(col1, col2, corr_matrix.loc[col1, col2])
               for col1 in num_cols for col2 in num_cols
               if col1 != col2 and abs(corr_matrix.loc[col1, col2]) > 0.7]

# Affichage
print("Variables fortement corrélées (>0.9) :")
for col1, col2, val in strong_corr:
    print(f"{col1} ↔ {col2} : corr = {val:.2f}")

Il faut maintenant identifier l'impact des Outliers sur l'analyse : 

### Outliers

In [ ]:
# Sélectionner variables numériques
num_cols = df.select_dtypes(include='number').columns.tolist()
num_cols.remove('target_is_fraud')  # enlever la target si elle est numérique

# Dictionnaire pour stocker le taux d'outliers par variable
outlier_stats = {}

for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    # Flag outlier
    df[col + '_outlier'] = ((df[col] < lower) | (df[col] > upper))
    
    # Taux d'outliers par classe
    taux_par_classe = df.groupby('target_is_fraud')[col + '_outlier'].mean() * 100
    outlier_stats[col] = taux_par_classe

# Affichage
for col, taux in outlier_stats.items():
    print(f"{col}: \n{taux}\n")

In [ ]:
# Dictionnaire pour stocker la différence
diff_outliers = {}

for col in num_cols:
    taux = df.groupby('target_is_fraud')[col + '_outlier'].mean()
    diff = taux.get(1,0) - taux.get(0,0)  # fraude - non-fraude
    diff_outliers[col] = diff

# Convertir en DataFrame et trier par valeur absolue croissante
diff_df = pd.DataFrame({
    'variable': diff_outliers.keys(),
    'diff_fraud_nonfraud': diff_outliers.values()
})

diff_df['abs_diff'] = diff_df['diff_fraud_nonfraud'].abs()
diff_df = diff_df.sort_values('abs_diff', ascending=False).reset_index(drop=True)

print(diff_df[['variable','diff_fraud_nonfraud','abs_diff']])

# 6. Variables catégorielles 

Les modèles ML ne comprennent pas les textes → il faut donc repérer afin de supprimer les colonnes concernées

In [ ]:
df.select_dtypes(include='object').nunique().sort_values(ascending=False)

In [ ]:
cols = ['device_type','channel','plan_type','browser']

for c in cols:
    stats = df.groupby(c).agg(
        fraud_rate=('target_is_fraud','mean'),
        pct=('target_is_fraud','count')
    )
    stats['pct'] = stats['pct'] / stats['pct'].sum() * 100
    print(f"\n=== {c} ===")
    print(stats.sort_values('fraud_rate', ascending=False))

In [169]:
df.groupby(df['secondary_email'].notna())['target_is_fraud'].mean()

secondary_email
False    0.030784
True     0.031321
Name: target_is_fraud, dtype: float64

# Datetime

In [170]:
df['signup_date'] = pd.to_datetime(df['signup_date'])

In [171]:
df['signup_day'] = df['signup_date'].dt.day
df['signup_month'] = df['signup_date'].dt.month
df['signup_year'] = df['signup_date'].dt.year

In [ ]:
fraude_by_month = df.groupby('signup_month')['target_is_fraud'].mean().sort_index()
print("\nTaux de fraude par mois :")
print(fraude_by_month)

## Conclusion
L'analyse des données clients révèle que la fraude chargeback est un phénomène rare mais détectable grâce à plusieurs signaux clairs.<br>

Les données ont été nettoyées et sont prêtes pour la modélisation. Deux variables ont été écartées car elles ne seraient disponibles qu'après la fraude, ce qui les rend inutilisables en conditions réelles.
<br>

Le principal défi reste le choix du meilleur compromis afin de minimiser le coût du model.